In [1]:
import pandas as pd
import boto3
from io import BytesIO

s3 = boto3.client(
    's3',
    endpoint_url='http://minio:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='admin123',
    verify=False
)

def load_parquet(bucket, key):
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_parquet(BytesIO(obj['Body'].read()))

wells = load_parquet('oil-data', 'wells.parquet')
production = load_parquet('oil-data', 'production.parquet')
telemetry = load_parquet('oil-data', 'well_telemetry.parquet')
targets = load_parquet('oil-data', 'well_targets.parquet')
pumps = load_parquet('oil-data', 'pumps.parquet')
pump_sensors = load_parquet('oil-data', 'pump_sensors.parquet')
failures = load_parquet('oil-data', 'pump_failures.parquet')
deliveries = load_parquet('oil-data', 'deliveries.parquet')
drivers = load_parquet('oil-data', 'drivers.parquet')
vehicles = load_parquet('oil-data', 'vehicles.parquet')
stations = load_parquet('oil-data', 'oil_stations.parquet')

print("Данные загружены")
print(f"wells: {len(wells)}")
print(f"production: {len(production)}")
print(f"telemetry: {len(telemetry)}")
print(f"targets: {len(targets)}")

Данные загружены
wells: 5
production: 150
telemetry: 48
targets: 90


In [2]:
print("Пропуски в production:")
print(production.isnull().sum())

production = production.fillna({'temperature': production['temperature'].median(),
                                 'pressure': production['pressure'].median(),
                                 'downtime_hours': 0})

daily_production = production.groupby('date').agg({
    'oil_ton': 'sum',
    'gas_m3': 'sum',
    'water_m3': 'sum',
    'energy_kwh': 'sum',
    'downtime_hours': 'mean'
}).reset_index()

print("Ежедневная агрегация:")
print(daily_production.head())

Пропуски в production:
prod_id            0
well_id            0
date               0
oil_ton            0
gas_m3             0
water_m3           0
energy_kwh         0
downtime_hours     0
temperature       30
pressure          30
dtype: int64
Ежедневная агрегация:
         date  oil_ton    gas_m3  water_m3  energy_kwh  downtime_hours
0  2025-10-01    717.5  197170.0     631.1     26730.0            5.54
1  2025-10-02    717.3  197200.0     630.5     26735.0            5.60
2  2025-10-03    719.2  197350.0     630.8     26770.0            5.56
3  2025-10-04    722.9  197810.0     627.4     26875.0            5.34
4  2025-10-05    721.0  197620.0     630.1     26810.0            5.50


In [3]:
telemetry['date'] = pd.to_datetime(telemetry['timestamp']).dt.date
daily_telemetry = telemetry.groupby(['well_id', 'date']).agg({
    'pump_speed_rpm': 'mean',
    'pump_current': 'mean',
    'pressure_in': 'mean',
    'pressure_out': 'mean',
    'temperature': 'mean',
    'vibration': 'mean',
    'oil_flow_rate': 'mean'
}).reset_index()

ml_data = daily_telemetry.merge(targets, on=['well_id', 'date'], how='inner')

ml_data = ml_data.merge(wells[['well_id', 'field_name', 'region']], on='well_id', how='left')

print(f"Датасет для ML: {len(ml_data)} записей")
print(ml_data.head())

Датасет для ML: 2 записей
   well_id        date  pump_speed_rpm  pump_current  pressure_in  \
0        1  2025-10-01     1472.541667     58.395833    95.070833   
1        2  2025-10-01     1430.666667     54.650000    91.287500   

   pressure_out  temperature  vibration  oil_flow_rate  daily_oil_ton  \
0    122.233333      88.2875   1.462500       8.808333          212.4   
1    115.516667      84.3500   1.441667       7.537500          185.9   

    field_name        region  
0  Severo-Ural  Khanty-Mansi  
1  Severo-Ural  Khanty-Mansi  


In [10]:
print(f"Всего записей в telemetry: {len(telemetry)}")
print(f"Уникальных well_id в telemetry: {telemetry['well_id'].unique()}")
print(f"Уникальных дат в telemetry: {telemetry['timestamp'].dt.date.unique()}")
print(f"\nВсего записей в targets: {len(targets)}")
print(f"Уникальных well_id в targets: {targets['well_id'].unique()}")
print(f"\nУникальных well_id в production: {production['well_id'].unique()}")
print(f"\nПервые 5 записей telemetry:")
print(telemetry.head())
print(f"\nПоследние 5 записей telemetry:")
print(telemetry.tail())

Всего записей в telemetry: 48
Уникальных well_id в telemetry: [1 2]
Уникальных дат в telemetry: [datetime.date(2025, 10, 1)]

Всего записей в targets: 90
Уникальных well_id в targets: [1 2 5]

Уникальных well_id в production: [1 2 3 4 5]

Первые 5 записей telemetry:
   record_id  well_id           timestamp  pump_speed_rpm  pump_current  \
0          1        1 2025-10-01 00:00:00          1470.0          58.2   
1          2        1 2025-10-01 01:00:00          1468.0          58.5   
2          3        1 2025-10-01 02:00:00          1472.0          58.0   
3          4        1 2025-10-01 03:00:00          1475.0          58.3   
4          5        1 2025-10-01 04:00:00          1473.0          58.1   

   pressure_in  pressure_out  temperature  vibration  oil_flow_rate  \
0         95.3         122.4         88.1        1.4            8.8   
1         95.1         122.1         88.2        1.5            8.9   
2         94.9         121.8         88.3        1.6            8.7  

In [4]:
daily_production.to_parquet('daily_production.parquet', index=False)
ml_data.to_parquet('ml_dataset.parquet', index=False)

s3.put_object(Bucket='oil-data', Key='daily_production.parquet', 
              Body=open('daily_production.parquet', 'rb').read())
s3.put_object(Bucket='oil-data', Key='ml_dataset.parquet', 
              Body=open('ml_dataset.parquet', 'rb').read())

print("Витрины сохранены в MinIO")

Витрины сохранены в MinIO


In [12]:
!pip install numpy==1.24.3 scipy==1.10.1


  Using cached numpy-1.24.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.6 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 636.9 kB/s eta 0:00:00a 0:00:01
Using cached numpy-1.24.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (17.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.1/34.1 MB 10.8 MB/s eta 0:00:0000:0100:01


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

features = ['pump_speed_rpm', 'pump_current', 'pressure_in', 'pressure_out',
            'temperature', 'vibration', 'oil_flow_rate']
X = ml_data[features]
y = ml_data['daily_oil_ton']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"Feature importance: {dict(zip(features, model.feature_importances_))}")

ml_data['predicted_oil_ton'] = model.predict(X)
ml_data[['well_id', 'date', 'daily_oil_ton', 'predicted_oil_ton']].to_parquet('predictions.parquet', index=False)
s3.put_object(Bucket='oil-data', Key='predictions.parquet', 
              Body=open('predictions.parquet', 'rb').read())
print("Прогнозы сохранены в MinIO")

MAE: 26.50
RMSE: 26.50
Feature importance: {'pump_speed_rpm': 0.0, 'pump_current': 0.0, 'pressure_in': 0.0, 'pressure_out': 0.0, 'temperature': 0.0, 'vibration': 0.0, 'oil_flow_rate': 0.0}
Прогнозы сохранены в MinIO
